A retail company receives daily updates for its product catalog, including new products, price changes, and discontinued items. Instead of overwriting the entire catalog or simply appending new records, they need to upsert the incoming data-updating existing products with the latest information and inserting new products-ensuring the catalog remains accurate and up-to-date in real-time.

**Querying Source**

In [0]:
%sql
SELECT * FROM pyspark_cata.source.products


## **UPSERTS**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [0]:
df= spark.sql("SELECT * FROM pyspark_cata.source.products")

# dedup
df = df.withColumn("dedup",row_number().over(Window.partitionBy("id").orderBy(desc("updatedDate"))))

df =  df.filter(col("dedup")==1).drop("dedup")

display(df)

In [0]:
# Creating delta object
from delta.tables import DeltaTable

if len(dbutils.fs.ls("/Volumes/pyspark_cata/source/db_volume/products_sink/")) > 0:
  dlt_obj = DeltaTable.forPath(spark,"/Volumes/pyspark_cata/source/db_volume/products_sink/")

  dlt_obj.alias("trgt").merge(
    df.alias("src"),
    "src.id = trgt.id")\
    .whenMatchedUpdateAll(condition = "src.updatedDate >= trgt.updatedDate")\
    .whenNotMatchedInsertAll()\
    .execute()
  print("This is upserting now")

else:
  df.write.format("delta")\
    .mode("Overwrite")\
    .save("/Volumes/pyspark_cata/source/db_volume/products_sink/")




In [0]:
%sql
SELECT * FROM delta.`/Volumes/pyspark_cata/source/db_volume/products_sink/`